# ETL Pipeline — Lecture 13
Reads raw CSVs -> cleans -> loads into PostgreSQL (SQLite as optianal also added) -> builds analytics tables

## Imports

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()


True

## Load raw data

In [2]:
def get_dfs(data_dir='data/raw'):
    files = ['customers.csv', 'orders.csv', 'order_items.csv', 'products.csv']
    return {
        name.replace('.csv', ''): pd.read_csv(os.path.join(data_dir, name))
        for name in files
    }

raw = get_dfs()
for name, df in raw.items():
    print(f'{name}: {df.shape}')

customers: (357, 4)
orders: (456, 4)
order_items: (914, 4)
products: (155, 4)


## Clean customers

NULL `customer_id`, drop row, can't identify without a PK;

Duplicate `customer_id`, keep first, PK must be unique;

Malformed email, set to `None` (keep row), customer is still real; don't delete them over a typo;

Unparseable `created_at`, drop row, timestamp is required for time-series analytics;

In [3]:
EMAIL_RE = r'^[\w.+\-]+@[\w\-]+\.[a-zA-Z]{2,}$'

def clean_customers(df):
    return (
        df.copy()
        .dropna(subset=['customer_id'])                                      # drop null PK
        .drop_duplicates(subset=['customer_id'], keep='first')               # unique PK
        .astype({'customer_id': 'int'})                                      # 1.0 -> 1
        .assign(
            email=lambda x: x['email'].where(                                # bad email -> None
                x['email'].str.match(EMAIL_RE, na=False), other=None
            ),
            created_at=lambda x: pd.to_datetime(x['created_at'], errors='coerce')
        )
        .dropna(subset=['created_at'])                                       # drop unparseable dates
        .set_index('customer_id')
    )

customers = clean_customers(raw['customers'])
customers.head(3)

,email,country,created_at
customer_id,,,
1,user1@example.com,DE,2024-01-04 23:17:00
2,user2@example.com,PL,2024-01-29 04:47:00
3,user3@example.com,DE,2024-03-27 23:57:00


## Clean products

Duplicate `product_id`, keep first, PK must be unique;

NULL name or category, drop row, unusable in analytics without a label;

Price ≤ 0, drop row, economically invalid;

In [4]:
def clean_products(df):
    return (
        df.copy()
        .drop_duplicates(subset=['product_id'], keep='first')                # unique PK (on PK only!)
        .dropna(subset=['name', 'category'])                                 # need label for analytics
        .loc[lambda x: x['price'] > 0]                                      # invalid prices out
        .astype({'product_id': 'int', 'price': 'float'})
        .set_index('product_id')
    )

products = clean_products(raw['products'])
products.head(3)

,name,category,price
product_id,,,
1001,Toys Product 1001,Toys,810.03
1002,Stationery Product 1002,Stationery,797.22
1003,Stationery Product 1003,Stationery,1246.05


## Clean orders



NULL `customer_id`, drop row, Orphan order;

Mixed-case status, lowercase, `COMPLETED` == `completed`;

Unknown status, drop row, can't classify the order;

`customer_id` not in customers, Drop row, FK violation — join would silently fail;

In [5]:
VALID_STATUSES = {'completed', 'pending', 'cancelled', 'returned'}

def clean_orders(df, valid_customer_ids):
    return (
        df.copy()
        .dropna(subset=['customer_id'])                                      # need a customer
        .astype({'customer_id': 'int'})
        .assign(
            order_status=lambda x: x['order_status'].str.lower().str.strip(),
            created_at=lambda x: pd.to_datetime(x['created_at'], errors='coerce')
        )
        .loc[lambda x: x['order_status'].isin(VALID_STATUSES)]              # known statuses only
        .loc[lambda x: x['customer_id'].isin(valid_customer_ids)]           # FK check
        .dropna(subset=['created_at'])
        .set_index('order_id')
    )

orders = clean_orders(raw['orders'], customers.index)
orders.head(3)

,customer_id,order_status,created_at
order_id,,,
5001,307,completed,2024-05-09 06:49:00
5002,71,completed,2024-04-16 00:31:00
5003,221,completed,2024-06-08 22:14:00


## Clean order items

Quantity ≤ 0, drop row, invalid transaction;

`order_id` not in orders, drop row, FK violation;

`product_id` not in products, drop row, can't compute revenue without price

In [6]:
def clean_order_items(df, valid_order_ids, valid_product_ids):
    return (
        df.copy()
        .loc[lambda x: x['quantity'] > 0]                                   # valid quantity
        .loc[lambda x: x['order_id'].isin(valid_order_ids)]                 # FK check
        .loc[lambda x: x['product_id'].isin(valid_product_ids)]             # FK check
        .astype({'order_item_id': 'int', 'quantity': 'int'})
        .set_index('order_item_id')
    )

order_items = clean_order_items(raw['order_items'], orders.index, products.index)
order_items.head(3)

,order_id,product_id,quantity
order_item_id,,,
1,5001,1142,5
2,5001,1068,3
3,5002,1061,1


## Summary of cleaning

In [7]:
for name, (before, after) in {
    'customers':   (raw['customers'],   customers),
    'products':    (raw['products'],    products),
    'orders':      (raw['orders'],      orders),
    'order_items': (raw['order_items'], order_items),
}.items():
    print(f'{name}: {len(before)} -> {len(after)}  (dropped {len(before) - len(after)})')

customers: 357 -> 352  (dropped 5)
products: 155 -> 150  (dropped 5)
orders: 456 -> 453  (dropped 3)
order_items: 914 -> 907  (dropped 7)


## Load into PostgreSQL

In [8]:
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME', 'etl_db')
DB_USER = os.getenv('DB_USER', 'etl')
DB_PASS = os.getenv('DB_PASS', 'etl_pass')

engine = create_engine(f'postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

def load(df, table):
    df.to_sql(table, engine, if_exists='replace', index=True, method='multi')
    print(f'  {len(df)} rows → {table}')

load(customers, 'clean_customers')
load(products, 'clean_products')
load(orders, 'clean_orders')
load(order_items, 'clean_order_items')


  352 rows → clean_customers
  150 rows → clean_products
  453 rows → clean_orders
  907 rows → clean_order_items


## Analytics tables
Build in SQL so the logic lives in the DB and can be queried directly.

In [9]:
analytics_sql = [
    """
    DROP TABLE IF EXISTS customer_summary;
    CREATE TABLE customer_summary AS
    SELECT
        c.customer_id, c.email, c.country,
        COUNT(DISTINCT o.order_id)            AS total_orders,
        COALESCE(SUM(oi.quantity * p.price), 0) AS total_spent
    FROM clean_customers c
    LEFT JOIN clean_orders      o  ON c.customer_id = o.customer_id
    LEFT JOIN clean_order_items oi ON o.order_id    = oi.order_id
    LEFT JOIN clean_products    p  ON oi.product_id = p.product_id
    GROUP BY c.customer_id, c.email, c.country;
    """,
    """
    DROP TABLE IF EXISTS product_performance;
    CREATE TABLE product_performance AS
    SELECT
        p.product_id, p.name, p.category,
        COALESCE(SUM(oi.quantity), 0)           AS total_quantity_sold,
        COALESCE(SUM(oi.quantity * p.price), 0) AS total_revenue
    FROM clean_products p
    LEFT JOIN clean_order_items oi ON p.product_id = oi.product_id
    GROUP BY p.product_id, p.name, p.category;
    """,
    """
    DROP TABLE IF EXISTS monthly_revenue;
    CREATE TABLE monthly_revenue AS
    SELECT
        DATE_TRUNC('month', o.created_at)       AS month,
        COUNT(DISTINCT o.order_id)              AS total_orders,
        COALESCE(SUM(oi.quantity * p.price), 0) AS revenue
    FROM clean_orders o
    LEFT JOIN clean_order_items oi ON o.order_id    = oi.order_id
    LEFT JOIN clean_products    p  ON oi.product_id = p.product_id
    GROUP BY DATE_TRUNC('month', o.created_at)
    ORDER BY month;
    """
]

with engine.connect() as conn:
    for q in analytics_sql:
        conn.execute(text(q))
    conn.commit()

print('Analytics tables created.')

Analytics tables created.


## Results

In [10]:
pd.read_sql('SELECT * FROM customer_summary ORDER BY total_spent DESC LIMIT 10', engine)

,customer_id,email,country,total_orders,total_spent
0,44,user44@example.com,UA,3,30203.22
1,10,user10@example.com,UA,3,25054.40
2,114,user114@example.com,NL,4,24156.38
3,335,user335@example.com,UA,5,22803.45
4,204,user204@example.com,ES,4,21745.49
5,33,user33@example.com,SE,4,21564.06
6,208,user208@example.com,DE,3,21129.75
7,111,user111@example.com,US,2,20527.57
8,323,user323@example.com,ES,3,20349.01
9,89,user89@example.com,CZ,4,20332.38


In [11]:
pd.read_sql('SELECT * FROM product_performance ORDER BY total_revenue DESC LIMIT 10', engine)

,product_id,name,category,total_quantity_sold,total_revenue
0,1122,Toys Product 1122,Toys,32.0,39296.00
1,1031,Furniture Product 1031,Furniture,26.0,34608.86
2,1081,Books Product 1081,Books,32.0,34213.76
3,1039,Books Product 1039,Books,24.0,33808.80
4,1147,Stationery Product 1147,Stationery,30.0,30539.10
5,1140,Toys Product 1140,Toys,32.0,29680.64
6,1002,Stationery Product 1002,Stationery,37.0,29497.14
7,1047,Electronics Product 1047,Electronics,20.0,28472.40
8,1036,Beauty Product 1036,Beauty,26.0,28351.70
9,1005,Books Product 1005,Books,19.0,27571.66


In [12]:
pd.read_sql('SELECT * FROM monthly_revenue', engine)

,month,total_orders,revenue
0,2024-04-01,114,556367.84
1,2024-05-01,131,488221.93
2,2024-06-01,115,424614.01
3,2024-07-01,93,383989.68


# SQLite

In [13]:
# SQLite - local database, no Docker required
sqlite_engine = create_engine('sqlite:///data/etl.db')

for name, df in {
    'clean_customers':   customers,
    'clean_products':    products,
    'clean_orders':      orders,
    'clean_order_items': order_items,
}.items():
    df.to_sql(name, sqlite_engine, if_exists='replace', index=True)
    print(f'  {len(df)} rows -> {name}')


  352 rows -> clean_customers
  150 rows -> clean_products
  453 rows -> clean_orders
  907 rows -> clean_order_items


In [14]:
# Analytics tables for SQLite.
# Difference from PostgreSQL: STRFTIME instead of DATE_TRUNC,
# each query is executed separately (SQLite does not support multi-statement)

sqlite_analytics = [
    'DROP TABLE IF EXISTS customer_summary',
    '''
    CREATE TABLE customer_summary AS
    SELECT
        c.customer_id, c.email, c.country,
        COUNT(DISTINCT o.order_id)              AS total_orders,
        COALESCE(SUM(oi.quantity * p.price), 0) AS total_spent
    FROM clean_customers c
    LEFT JOIN clean_orders      o  ON c.customer_id = o.customer_id
    LEFT JOIN clean_order_items oi ON o.order_id    = oi.order_id
    LEFT JOIN clean_products    p  ON oi.product_id = p.product_id
    GROUP BY c.customer_id, c.email, c.country
    ''',
    'DROP TABLE IF EXISTS product_performance',
    '''
    CREATE TABLE product_performance AS
    SELECT
        p.product_id, p.name, p.category,
        COALESCE(SUM(oi.quantity), 0)           AS total_quantity_sold,
        COALESCE(SUM(oi.quantity * p.price), 0) AS total_revenue
    FROM clean_products p
    LEFT JOIN clean_order_items oi ON p.product_id = oi.product_id
    GROUP BY p.product_id, p.name, p.category
    ''',
    'DROP TABLE IF EXISTS monthly_revenue',
    '''
    CREATE TABLE monthly_revenue AS
    SELECT
        STRFTIME('%Y-%m-01', o.created_at) AS month,
        COUNT(DISTINCT o.order_id)            AS total_orders,
        COALESCE(SUM(oi.quantity * p.price), 0) AS revenue
    FROM clean_orders o
    LEFT JOIN clean_order_items oi ON o.order_id    = oi.order_id
    LEFT JOIN clean_products    p  ON oi.product_id = p.product_id
    GROUP BY STRFTIME('%Y-%m-01', o.created_at)
    ORDER BY month
    ''',
]

with sqlite_engine.connect() as conn:
    for q in sqlite_analytics:
        conn.execute(text(q))
    conn.commit()

print('SQLite analytics tables created.')


SQLite analytics tables created.


In [ ]:
print('Top 10 customers by spending')
pd.read_sql('SELECT * FROM customer_summary ORDER BY total_spent DESC LIMIT 10', sqlite_engine)


Top 10 customers by spending

Top 10 products by revenue

Monthly revenue


,month,total_orders,revenue
0,2024-04-01,114,556367.84
1,2024-05-01,131,488221.93
2,2024-06-01,115,424614.01
3,2024-07-01,93,383989.68


In [ ]:
print('\nTop 10 products by revenue')
pd.read_sql('SELECT * FROM product_performance ORDER BY total_revenue DESC LIMIT 10', sqlite_engine)

In [ ]:
print('\nMonthly revenue')
pd.read_sql('SELECT * FROM monthly_revenue', sqlite_engine)
